# Correlation-length EP Spectrum — MultiScaleCNEEP1D Notebook

이 노트북은 `AMB_1D` 시스템에서 **거리(correlation length) $r$에 따른 국소적 Entropy Production(EP) 발생량**을
스펙트럼처럼 분석하기 위한 `MultiScaleCNEEP1D` 모델을 훈련합니다.

**핵심 아이디어**:
- 각 거리 $k \in \{1, \ldots, K\}$ 마다 독립적인 병렬 브랜치(Parallel Branch)가
  `MaskedConv1d`로 양 끝과 중심 정보만 융합하고,
  1×1 Conv (Point-wise MLP)로 Local EP Map을 생성합니다.
- Time-reversal antisymmetry 연산은 Local EP Map 차원에서 수행합니다.
- 각 브랜치의 Global Average Pooling 결과가 $J_k$ (거리 $k$에서의 EP)를 나타냅니다.
- 전체 EP = $\sum_k J_k$로 합산하여 α-NEEP loss로 훈련합니다.


In [ ]:
import sys
import os

CNEEP_V2_ROOT = os.path.abspath('/home/user1/CNEEP_v2')

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)


In [ ]:
sys.path.append(CNEEP_V2_ROOT)
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data', 'AMB'))
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'utils'))
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'models'))

from argparse import Namespace
import numpy as np
import torch
from datetime import datetime
from utils.sampler import CartesianSeqSampler
from tqdm import tqdm
import matplotlib.pyplot as plt
from generate_trajectories_1d import ActiveModelB1D


## 0. Hyperparameters

In [ ]:
#
# Hyper parameters
#
opt = Namespace()

# dataset parameter
n_trajs = 1000
n_steps = 1000
burn_in = 10000
opt.device = 'cuda' if torch.cuda.is_available() else 'cpu'

# alpha-NEEP
opt.alpha     = -0.5
opt.lam       = 0.0
opt.threshold = 0.01

opt.positional  = False
opt.periodic    = True

# training
opt.n_iter           = 5000
opt.train_batch_size = 4096
opt.test_batch_size  = 4096
opt.video_batch_size = 256
opt.lr               = 1e-3
opt.wd               = 1e-3
opt.scalar           = 1
opt.input_scalar     = 1
opt.loss_scalar      = 1e2
opt.clip_norm        = 1

opt.record_freq = 1000
opt.seed        = 3

# MultiScaleCNEEP1D specific
opt.n_channel    = 32
opt.n_hidden     = 2
opt.max_distance = 10     # K: maximum correlation distance
opt.beta         = 1.0    # coefficient for antisymmetric term
opt.input_shape  = (256,) # AMB grid size
opt.seq_len      = 2
opt.val_ratio    = 0.2
opt.time_step    = 0.01   # dt

# AMB model parameters
kwargs = {
    'Lx': 256,
    'dx': 1,
    'a': 0.125,
    'b': 0.125,
    'kappa': 8.0,
    'lam': 20.0,
    'D': 0.1,
    'dt': 0.01,
    'smooth': True,
    'backend': 'torch',
    'use_gpu': torch.cuda.is_available(),
    'bc': 'periodic',
    'epr_mu_active_only': False
}

torch.manual_seed(opt.seed)

# results folder
result_folder = os.path.join(CNEEP_V2_ROOT, 'results')
current_result_folder = os.path.join(
    result_folder, f"Corr1D-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}")
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')

print(f'Device: {opt.device}')
print(f'Results: {current_result_folder}')


## 1. Generate AMB trajectory (train)

In [ ]:
train_seed = 42
np.random.seed(train_seed)
if kwargs['backend'] == 'torch':
    torch.manual_seed(train_seed)

model_amb = ActiveModelB1D(**kwargs)

print(f'[INFO] Generating TRAIN trajectory (n_trajs={n_trajs}, burn_in={burn_in}, n_steps={n_steps})')
trajectory = model_amb.generate_trajectories(
    n_trajectories=n_trajs, n_steps=n_steps, burn_in=burn_in
)
print(f'[INFO] Trajectory shape: {trajectory.shape}')

traj_train = trajectory
opt.M = traj_train.shape[0]
opt.L = traj_train.shape[1]
print(f'[INFO] Train data: {traj_train.shape}  (M={opt.M}, L={opt.L})')


## 1-b. Generate AMB trajectory (test, different seed)

In [ ]:
test_seed = 123
np.random.seed(test_seed)
if kwargs['backend'] == 'torch':
    torch.manual_seed(test_seed)

model_amb_test = ActiveModelB1D(**kwargs)

opt.M_test = 1
opt.L_test = 10000

print(f'[INFO] Generating TEST trajectory (n_trajs={opt.M_test}, burn_in={burn_in}, n_steps={opt.L_test})')
trajectory_test = model_amb_test.generate_trajectories(
    n_trajectories=opt.M_test, n_steps=opt.L_test, burn_in=burn_in
)
print(f'[INFO] Test trajectory shape: {trajectory_test.shape}')

traj_test = trajectory_test
opt.M_test = traj_test.shape[0]
opt.L_test = traj_test.shape[1]
print(f'[INFO] Test data: {traj_test.shape}  (M_test={opt.M_test}, L_test={opt.L_test})')


## 2. Prepare video tensors (train & test)

In [ ]:
#
# Convert density field to video tensor: (M, L, 1, Lx)
#
train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

traj_train_new = traj_train[:train_val_split_idx]
traj_val = traj_train[train_val_split_idx:]

train_video = torch.from_numpy(traj_train_new).float().to(opt.device)
train_video = train_video.unsqueeze(2)   # (M_train, L, 1, Lx)

val_video = torch.from_numpy(traj_val).float().to(opt.device)
val_video = val_video.unsqueeze(2)       # (M_val, L, 1, Lx)

test_video = torch.from_numpy(traj_test).float().to(opt.device)
test_video = test_video.unsqueeze(2)     # (M_test, L_test, 1, Lx)

# Use training data for normalization
mean = torch.mean(train_video)
std  = torch.std(train_video)
transform = lambda x: (x - mean) * opt.input_scalar / std

print(f'Train video tensor: {train_video.shape}')
print(f'Val video tensor:   {val_video.shape}')
print(f'Test video tensor:  {test_video.shape}')
print(f'Mean: {mean:.4f}, Std: {std:.4f}')


## 3. Build and train model

In [ ]:
from models.NEEP_Corr_1D import MultiScaleCNEEP1D as CNEEP

model = CNEEP(opt)
model = model.to(opt.device)
optim = torch.optim.Adam(model.parameters(), opt.lr, weight_decay=opt.wd)

train_sampler = CartesianSeqSampler(
    M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)
val_sampler = CartesianSeqSampler(
    M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Max distance (K): {opt.max_distance}')
print(f'Hidden channels: {opt.n_channel}')
print(f'Hidden layers per branch: {opt.n_hidden}')
print(model)


In [ ]:
#
# Training loop (MultiScaleCNEEP1D)
#
# model output: [B, K] — EP contribution at each distance k
# total EP per sample = sum over all distances
#
from livelossplot import PlotLosses

train_losses = []
valid_losses = []

liveloss = PlotLosses()
smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None

for it in tqdm(range(1, opt.n_iter + 1)):
    # ── Train step ──
    model.train()
    batch = next(train_sampler)

    b0 = batch[0].to(train_video.device)
    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]
    x = transform(torch.cat(slices, dim=1).float().to(opt.device))

    # model forward: [B, K]
    J_all = model(x) / opt.scalar   # [B, K]  EP at each distance

    # Total EP per sample: sum over all correlation distances, scaled by volume
    vol = kwargs['Lx'] * kwargs['dx']
    ent_production = J_all.sum(dim=1) * vol  # [B]

    optim.zero_grad()

    # alpha-NEEP loss on total EP
    if opt.alpha == 0:
        loss = (- ent_production + (torch.exp(-ent_production) - 1)).mean()
    else:
        loss = (- (torch.exp(opt.alpha * ent_production) - 1) / opt.alpha
            + (torch.exp(-(1 + opt.alpha) * ent_production) - 1) / (1 + opt.alpha)).mean()

    (loss * opt.loss_scalar).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)
    optim.step()

    train_losses.append(loss.item())

    # ── Validation & checkpoint ──
    if it % opt.record_freq == 0 or it == 1:
        model.eval()
        val_loss_acc = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_sampler:
                vb0 = vb[0].to(val_video.device)
                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]
                vx = transform(torch.cat(vslices, dim=1).float().to(opt.device))

                vJ = model(vx) / opt.scalar
                vol = kwargs['Lx'] * kwargs['dx']
                v_ep = vJ.sum(dim=1) * vol

                if opt.alpha == 0:
                    vloss = (- v_ep + (torch.exp(-v_ep) - 1)).sum().item()
                else:
                    vloss = (- (torch.exp(opt.alpha * v_ep) - 1) / opt.alpha
                        + (torch.exp(-(1 + opt.alpha) * v_ep) - 1) / (1 + opt.alpha)).sum().item()
                val_loss_acc += vloss
                n_val += vx.shape[0]

        avg_val = val_loss_acc / n_val
        valid_losses.append(avg_val)

        # Save checkpoint
        state = {
            'settings': opt.__dict__,
            'state_dict': model.state_dict(),
            'optimizer': optim.state_dict(),
            'iteration': it,
        }
        torch.save(state, current_checkpoint_path)

        # Update livelossplot
        if smooth_train_loss is None:
            smooth_train_loss = loss.item()
            smooth_val_loss = avg_val
        else:
            smooth_train_loss = smoothing * smooth_train_loss + (1 - smoothing) * loss.item()
            smooth_val_loss = smoothing * smooth_val_loss + (1 - smoothing) * avg_val
        
        liveloss.update({'train_loss': smooth_train_loss, 'val_loss': smooth_val_loss})
        liveloss.send()

print('Training finished.')
print(f'Checkpoint: {current_checkpoint_path}')


## 4. Training curves

In [ ]:
#
# Training curves
#
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(train_losses[100:]); axes[0].set_title('Train Loss')
axes[1].plot(valid_losses); axes[1].set_title('Valid Loss')
for ax in axes: ax.set_xlabel('Iteration')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/training_curves.png', dpi=150)
plt.show()


## 5. Ground Truth EPR vs Predicted EPR

In [ ]:
#
# Calculate Ground Truth EPR for the test trajectory
#
print('[INFO] Calculating GT EPR...')
gt_epr_maps = []
for i in tqdm(range(opt.M_test)):
    # model_amb_test.epr(phi) returns EPR density map [L, Lx]
    epr_map = model_amb_test.epr(traj_test[i]) # [L, Lx]
    gt_epr_maps.append(epr_map)

gt_epr_maps = np.stack(gt_epr_maps) # [M_test, L, Lx]
gt_total_epr = gt_epr_maps.sum(axis=-1) * model_amb_test.dx # [M_test, L] -> total EPR at each time step
gt_total_epr = gt_total_epr.flatten()

print(f'GT EPR shape: {gt_total_epr.shape}')


In [ ]:
#
# Compare total Predicted EP (sum of all J_k) with GT EPR
#
model.eval()
pred_total_ep = []

test_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.cat(slices, dim=1).float().to(opt.device))
        J_all = model(x) / opt.scalar  # [B, K]
        total_ep = J_all.sum(dim=1)     # [B]
        pred_total_ep.append(total_ep.cpu().numpy())

pred_total_ep = np.concatenate(pred_total_ep) # [N_total]
vol = kwargs['Lx'] * kwargs['dx']
pred_total_ep = pred_total_ep * vol

min_len = min(len(gt_total_epr), len(pred_total_ep))
dt = kwargs['dt']
time_axis = np.arange(min_len) * dt

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# (a) Instantaneous EPR
axes[0].plot(time_axis, gt_total_epr[:min_len], lw=0.5, alpha=0.6, label='GT EPR')
axes[0].plot(time_axis, pred_total_ep[:min_len] / dt, lw=0.5, alpha=0.6, label='Pred EP/dt')
axes[0].set_ylabel('EPR')
axes[0].legend()
axes[0].set_title('Instantaneous EPR Comparison')

# (b) Cumulative EP
axes[1].plot(time_axis, np.cumsum(gt_total_epr[:min_len] * dt), label='GT Cumul EP')
axes[1].plot(time_axis, np.cumsum(pred_total_ep[:min_len]), label='Pred Cumul EP')
axes[1].set_ylabel('Cumulative EP')
axes[1].legend()

# (c) Running Average
window = 100
gt_smooth = np.convolve(gt_total_epr[:min_len], np.ones(window)/window, mode='same')
pred_smooth = np.convolve(pred_total_ep[:min_len]/dt, np.ones(window)/window, mode='same')
axes[2].plot(time_axis, gt_smooth, label='GT (Smooth)')
axes[2].plot(time_axis, pred_smooth, label='Pred (Smooth)')
axes[2].set_ylabel('EPR (Running Avg)')
axes[2].set_xlabel('Time')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{current_result_folder}/epr_timeseries.png', dpi=150)
plt.show()

print(f'GT mean EPR:   {gt_total_epr[:min_len].mean():.6e}')
print(f'Pred mean EPR:  {(pred_total_ep[:min_len]/dt).mean():.6e}')


## 6. Spatial EP Map & Ensemble Average

In [ ]:
#
# Visualize Local EP Map
#
model.eval()

# Take a sample batch
test_sampler_one = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, 1,
    device=opt.device, train=False)
ens_idx, traj_idx = next(test_sampler_one)
b0 = ens_idx.to(test_video.device)
slices = [test_video[(b0, traj_idx[i].to(test_video.device))] for i in range(opt.seq_len)]
x = transform(torch.cat(slices, dim=1).float().to(opt.device))

with torch.no_grad():
    # return_maps=True returns [B, K, L]
    maps = model(x, return_maps=True) / opt.scalar # [B, K, L]

sample_idx = 0
dt = kwargs['dt']
phi_t = x[sample_idx, 0].cpu().numpy() # [L]
pred_map_k = maps[sample_idx].cpu().numpy() / (kwargs['dx'] * dt) # [K, L]
pred_total_map = pred_map_k.sum(axis=0) # [L]
gt_map = gt_epr_maps[ens_idx[sample_idx].item(), traj_idx[0].item()]

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
axes[0].plot(phi_t, color='black', lw=1.5)
axes[0].set_ylabel(r'Density $\phi$')
axes[0].set_title('Input State')

axes[1].plot(gt_map, label='GT EPR Map', color='steelblue', lw=2)
axes[1].plot(pred_total_map, label='Pred Total EP Map', color='crimson', lw=2, linestyle='--')
axes[1].set_ylabel('EPR Density')
axes[1].set_title('Total EP Map: GT vs Predicted')
axes[1].legend()

for k in range(opt.max_distance + 1):
    axes[2].plot(pred_map_k[k], label=f'k={k}', lw=1)
axes[2].set_ylabel('EPR Density')
axes[2].set_xlabel('Space $x$')
axes[2].set_title('Local EP Spectrum Map by Distance $k$')
axes[2].legend(loc='upper right', ncol=2)
plt.tight_layout()
plt.savefig(f'{current_result_folder}/local_ep_map.png', dpi=150)
plt.show()


In [ ]:
#
# Ensemble Averaged EP Map
#
print('[INFO] Calculating Ensemble Averaged Map...')
all_maps = []
test_sampler_batch = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

with torch.no_grad():
    for batch in tqdm(test_sampler_batch):
        b0 = batch[0].to(test_video.device)
        vslices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        vx = transform(torch.cat(vslices, dim=1).float().to(opt.device))
        m = model(vx, return_maps=True) / opt.scalar # [B, K, L]
        all_maps.append(m.cpu().numpy())

dt = kwargs['dt']
all_maps = np.concatenate(all_maps, axis=0) / (kwargs['dx'] * dt) # [N, K, L]
ensemble_map_k = all_maps.mean(axis=0) # [K, L]
ensemble_pred_total = ensemble_map_k.sum(axis=0) # [L]
ensemble_gt = gt_epr_maps.mean(axis=(0, 1)) # [L]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(ensemble_gt, label='GT Ensemble EPR Map', color='steelblue', lw=2)
axes[0].plot(ensemble_pred_total, label='Pred Ensemble Total EP Map', color='crimson', lw=2, linestyle='--')
axes[0].set_ylabel('Mean EPR Density')
axes[0].set_title('Ensemble Averaged Total EP Map: GT vs Predicted')
axes[0].legend()
axes[0].set_ylim(bottom=0)

for k in range(opt.max_distance + 1):
    axes[1].plot(ensemble_map_k[k], label=f'k={k}', lw=1)
axes[1].set_ylabel('Mean EPR Density')
axes[1].set_xlabel('Space $x$')
axes[1].set_title('Ensemble Averaged EP Spectrum Map by Distance $k$')
axes[1].legend(loc='upper right', ncol=2)
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(f'{current_result_folder}/ensemble_ep_map.png', dpi=150)
plt.show()


## 7. EP Spectrum — EP decomposition by correlation distance

각 거리 $k$에서의 평균 EP 기여도 $\langle J_k \rangle$를 시각화합니다.

In [ ]:
#
# EP spectrum: mean J_k for each distance k
#
model.eval()

all_J = []  # collect [B, K] arrays

test_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.cat(slices, dim=1).float().to(opt.device))
        J = model(x) / opt.scalar  # [B, K]
        all_J.append(J.cpu().numpy())

all_J = np.concatenate(all_J, axis=0)  # [N_total, K]
mean_J = all_J.mean(axis=0)             # [K]
std_J  = all_J.std(axis=0)              # [K]

distances = np.arange(0, opt.max_distance + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: mean EP at each distance
axes[0].bar(distances, mean_J / kwargs['dt'], yerr=std_J / kwargs['dt'] / np.sqrt(len(all_J)),
            capsize=3, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Correlation distance $k$')
axes[0].set_ylabel('$\\langle J_k \\rangle / dt$')
axes[0].set_title('EP Spectrum: EP rate by correlation distance')
axes[0].set_xticks(distances)

# Cumulative sum
cum_J = np.cumsum(mean_J)
axes[1].plot(distances, cum_J / kwargs['dt'], 'o-', color='darkorange')
axes[1].set_xlabel('Correlation distance $k$')
axes[1].set_ylabel('$\\sum_{i=1}^{k} \\langle J_i \\rangle / dt$')
axes[1].set_title('Cumulative EP rate')
axes[1].set_xticks(distances)

plt.tight_layout()
plt.savefig(f'{current_result_folder}/ep_spectrum.png', dpi=150)
plt.show()

print(f'Total estimated EP rate: {mean_J.sum() / kwargs["dt"]:.6e}')
for k in range(opt.max_distance + 1):
    print(f'  k={k}: J_k / dt = {mean_J[k] / kwargs["dt"]:.6e}  '
          f'({100 * mean_J[k] / mean_J.sum():.1f}%)')
